# 03. Dimensionality reduction

Parul. PCA, LDA, and t-SNE on the combined 8206-d features from `02_feature_extraction.ipynb`. Loads files the same way as `_load_features_to_your_usecase.ipynb`.


In [ ]:
# Colab setup
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install numpy pandas matplotlib scikit-learn joblib
    if not os.path.exists("/content/MIDS-281-project/.git"):
        !git clone --depth 1 https://github.com/sreeramchennakrish/MIDS-281-project.git /content/MIDS-281-project
    %cd /content/MIDS-281-project/notebooks

print("cwd:", os.getcwd())


## Load combined features


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

FEATURE_DIR = Path("../features").resolve()
OUTPUT_DIR = Path("../outputs/dimred").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with np.load(FEATURE_DIR / "combined_weighted_features.npz") as data:
    X_train = data["X_train"]
    X_val = data["X_val"]
    y_train = data["y_train"]
    y_val = data["y_val"]
    classes = np.asarray(data["classes"])
    feature_groups = data["feature_groups"]
    feature_dimensions = data["feature_dimensions"]
    feature_weights = data["feature_weights"]

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Classes:", classes)
for name, dim, weight in zip(feature_groups, feature_dimensions, feature_weights):
    print(f"{name}: {dim} features, weight={weight}")


## Optional: individual feature groups


In [ ]:
# Optional: individual feature groups (same loader as _load_features_to_your_usecase.ipynb)
feature_names = ["rgb", "hsv", "lbp", "hog"]
features = {}
for name in feature_names:
    with np.load(FEATURE_DIR / f"{name}_features.npz") as data:
        features[name] = {
            "X_train": data["X_train"],
            "X_val": data["X_val"],
            "y_train": data["y_train"],
            "y_val": data["y_val"],
        }
    print(name.upper(), "Train:", features[name]["X_train"].shape, "Val:", features[name]["X_val"].shape)


## PCA


In [ ]:
# PCA is fit on train only, then applied to val.
# Combined features are already standardized per group; a light scaler still helps PCA.
pca_scaler = StandardScaler()
X_train_z = pca_scaler.fit_transform(X_train)
X_val_z = pca_scaler.transform(X_val)

pca_full = PCA(random_state=42)
pca_full.fit(X_train_z)
cum = np.cumsum(pca_full.explained_variance_ratio_)
n90 = int(np.searchsorted(cum, 0.90) + 1)
n95 = int(np.searchsorted(cum, 0.95) + 1)
print(f"Components for 90% variance: {n90}")
print(f"Components for 95% variance: {n95}")

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, min(len(cum), 400) + 1), cum[:400])
plt.axhline(0.90, ls="--", c="gray")
plt.axhline(0.95, ls="--", c="gray")
plt.xlabel("PCA components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA on combined 8206-d features (train)")
plt.tight_layout()
plt.show()

pca2 = PCA(n_components=2, random_state=42)
Z_train = pca2.fit_transform(X_train_z)
Z_val = pca2.transform(X_val_z)
print("PCA-2 explained variance:", pca2.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, Z, y, title in (
    (axes[0], Z_train, y_train, "Train PCA"),
    (axes[1], Z_val, y_val, "Val PCA"),
):
    for i, name in enumerate(classes):
        m = y == i
        ax.scatter(Z[m, 0], Z[m, 1], s=12, alpha=0.6, label=name)
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend()
plt.tight_layout()
plt.show()

np.savez_compressed(
    OUTPUT_DIR / "pca_projections.npz",
    X_train_pca2=Z_train,
    X_val_pca2=Z_val,
    n90=n90,
    n95=n95,
)
print("Saved:", OUTPUT_DIR / "pca_projections.npz")


## LDA


In [ ]:
# LDA: at most n_classes-1 = 2 directions.
lda = LDA(n_components=2)
L_train = lda.fit_transform(X_train_z, y_train)
L_val = lda.transform(X_val_z)
print("LDA explained variance ratio:", lda.explained_variance_ratio_)
print("Val accuracy from LDA decision:", lda.score(X_val_z, y_val))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, Z, y, title in (
    (axes[0], L_train, y_train, "Train LDA"),
    (axes[1], L_val, y_val, "Val LDA"),
):
    for i, name in enumerate(classes):
        m = y == i
        ax.scatter(Z[m, 0], Z[m, 1], s=12, alpha=0.6, label=name)
    ax.set_title(title)
    ax.set_xlabel("LD1")
    ax.set_ylabel("LD2")
    ax.legend()
plt.tight_layout()
plt.show()


## t-SNE (PCA-50 then t-SNE-2)


In [ ]:
# t-SNE is slow in 8206-d, so compress with PCA first (fit on train).
pca50 = PCA(n_components=50, random_state=42)
X_train_50 = pca50.fit_transform(X_train_z)
X_val_50 = pca50.transform(X_val_z)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=42,
)
# Joint embedding of train+val for a single picture; labels still come from the split.
X_all_50 = np.vstack([X_train_50, X_val_50])
y_all = np.concatenate([y_train, y_val])
split = np.array(["train"] * len(y_train) + ["val"] * len(y_val))
T = tsne.fit_transform(X_all_50)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, mask_name in ((axes[0], "train"), (axes[1], "val")):
    sel = split == mask_name
    Z = T[sel]
    y = y_all[sel]
    for i, name in enumerate(classes):
        m = y == i
        ax.scatter(Z[m, 0], Z[m, 1], s=12, alpha=0.6, label=name)
    ax.set_title(f"t-SNE ({mask_name})")
    ax.legend()
plt.tight_layout()
plt.show()


## Notes


LDA is supervised and uses the 3 class labels, so the 2D plot is expected to separate Fire / Smoke / Neutral more cleanly than PCA. PCA and t-SNE are unsupervised views of the same 8206-d combined features (RGB+HSV+LBP+HOG). Use the 90%/95% PCA counts in the next notebook (`05_classifier_logreg_rf.ipynb`) if you want a reduced classifier.
